# Instrument Data Compilation
This notebook compiles raw instrument data into Parquet files.
This script only runs from within Meteowiss, because of the data store. It compiles raw data files stored on MeteoSwiss disk into parquet files. Incoming raw data files are organized into folders by year and month, and a statistic is computed showing the number of recently incoming files.

joerg.klausen@meteoswiss.ch

In [1]:
from pathlib import Path
from processing.ae33 import AE33
from processing.fidas import Fidas
from processing.g2401 import G2401
from processing.meteo import Meteo
from processing.neph import Neph
from processing.thermo import Thermo
from processing.hmp110 import HMP110

from toolbox.utils import load_config

##############################
# Process MKN incoming files #
##############################
# read configuration
mkn = load_config(config_file="mch-mkn.yml")
target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

processors = {
    "49c": Thermo(name="tei49c"),
    "49i": Thermo(name="tei49i"),
    "g2401": G2401(),
    "ne300": Neph(name="ne300"),
    "ae33": AE33(),
    "hmp110-inlet": HMP110(name="hmp110-inlet"),
    "hmp110-ae33": HMP110(name="hmp110-ae33"),
    "fidas": Fidas(),
    "meteo": Meteo(name="vrxa00"),
}

for name, processor in processors.items():
    print(f"▶ Processing {name.upper()} ...", flush=True)
    processor.compile_to_parquet(
        source=Path(mkn['root']) / mkn['branches']['incoming'] / name,
        target=target / "level1" / "mkn",
        archive=Path(mkn['root']) / mkn['branches']['archive'] / name,
        issues=Path(mkn['root']) / mkn['branches']['issues'] / name,
        split=mkn[name]['split_parquet']
    )

▶ Processing 49C ...
2026-09-07T06:42:47, INFO, tei49c, Processing 49c-2026090705.zip
2026-09-07T06:42:47, INFO, tei49c, 49c-2026090705.zip::49c-2026090705.csv: detected model=tei49c layout=modern header=True
2026-09-07T06:42:47, INFO, tei49c, 49c-2026090705.zip::49c-2026090705.csv: extracted 58 row(s), 16 column(s)
2026-09-07T06:42:47, INFO, tei49c, ✔ Written 58 new rows to /product_data/data/pay/Kenya/git/gawkenyadata/level1/mkn/2026/09/tei49c.parquet
2026-09-07T06:42:47, INFO, tei49c, ✔ Parquet compilation completed. Total rows: 58
▶ Processing 49I ...
2026-09-07T06:42:47, INFO, tei49i, Processing 49i-2026090705.zip
2026-09-07T06:42:47, INFO, tei49i, 49i-2026090705.zip::49i-2026090705.csv: detected model=tei49i layout=modern header=True
2026-09-07T06:42:47, INFO, tei49i, 49i-2026090705.zip::49i-2026090705.csv: extracted 58 row(s), 17 column(s)
2026-09-07T06:42:47, INFO, tei49i, ✔ Written 58 new rows to /product_data/data/pay/Kenya/git/gawkenyadata/level1/mkn/2026/09/tei49i.parquet
2

In [2]:
from pathlib import Path

from processing.ae31 import AE31
from processing.avo import AVO
from processing.fidas import Fidas
from processing.hmp110 import HMP110
from processing.neph import Neph
from processing.thermo import Thermo

from toolbox.utils import load_config


##############################
# Process NRB incoming files #
##############################

nrb = load_config(config_file="mch-nrb.yml")
config = nrb["nrb-aq"]

log_file = str(Path("logs") / "nrb.log")

data_root = Path(
    "/product_data/data/pay/Kenya/git/gawkenyadata"
)

level1_root = Path(
    str(
        config.get(
            "level1_root",
            data_root / "level1",
        )
    )
)

default_target = str(
    config.get(
        "default_target",
        "nrb",
    )
)

raw_root = Path(str(config["root"]))
branches = config["branches"]

processors = {
    "49i": Thermo(
        name="49i",
        log_file=log_file,
    ),
    "fidas": Fidas(
        log_file=log_file,
    ),
    "aurora3000": Neph(
        name="aurora3000",
        log_file=log_file,
    ),
    "ae31": AE31(
        log_file=log_file,
    ),
    "hmp110-ae31": HMP110(
        name="hmp110-ae31",
        log_file=log_file,
    ),
    "hmp110-inlet": HMP110(
        name="hmp110-inlet",
        log_file=log_file,
    ),
    "hmp110-lab": HMP110(
        name="hmp110-lab",
        log_file=log_file,
    ),
    "avo-roof": AVO(
        name="avo-roof",
        log_file=log_file,
    ),
    "avo-garden": AVO(
        name="avo-garden",
        log_file=log_file,
    ),
    "avo-mogogosiek": AVO(
        name="avo-mogogosiek",
        log_file=log_file,
    ),
    "avo-huduma": AVO(
        name="avo-huduma",
        log_file=log_file,
    ),
}

processor_failures: dict[str, list[str]] = {}


def record_failure(
    processor_name: str,
    phase: str,
    error: Exception,
) -> None:
    """Record and display a non-fatal processor failure."""
    message = (
        f"{phase}: {type(error).__name__}: {error}"
    )

    processor_failures.setdefault(
        processor_name,
        [],
    ).append(message)

    print(
        f"✖ {processor_name.upper()} {message}",
        flush=True,
    )


for name, processor in processors.items():
    instrument_config = config[name]

    target_name = str(
        instrument_config.get(
            "target",
            default_target,
        )
    ).strip()

    if not target_name:
        raise ValueError(
            f"Empty target configured for {name!r}"
        )

    target_dir = level1_root / target_name

    incoming_dir = (
        raw_root
        / str(branches["incoming"])
        / name
    )

    archive_dir = (
        raw_root
        / str(branches["archive"])
        / name
    )

    issues_dir = (
        raw_root
        / str(branches["issues"])
        / name
    )

    split = str(
        instrument_config["split_parquet"]
    )

    read_archive = bool(
        instrument_config.get(
            "read_archive",
            False,
        )
    )

    print(
        f"\n▶ Processing {name.upper()} "
        f"-> {target_dir}",
        flush=True,
    )

    # AVO exports may already have been archived by an earlier run
    # that used a different output layout. Process archive as a
    # read-only input source first so those records can rebuild the
    # correctly located level1 products.
    #
    # archive=None prevents archived files from being moved again.
    # issues=None prevents archived files from being mutated if an
    # individual archived export cannot be parsed.
    if read_archive:
        print(
            f"  Reading archived source files: {archive_dir}",
            flush=True,
        )

        try:
            processor.compile_to_parquet(
                source=archive_dir,
                target=target_dir,
                archive=None,
                issues=None,
                split=split,
            )

        except Exception as err:
            record_failure(
                name,
                "archive-source compilation failed",
                err,
            )

            logger = getattr(
                processor,
                "logger",
                None,
            )

            if logger is not None:
                try:
                    logger.exception(
                        "Archive-source compilation failed for %s; "
                        "continuing with incoming files",
                        name,
                    )
                except Exception:
                    pass

    # Process incoming files independently of the archive-source pass.
    # Successful parquet writing must not depend on whether source
    # files can subsequently be moved into archive.
    print(
        f"  Reading incoming source files: {incoming_dir}",
        flush=True,
    )

    try:
        processor.compile_to_parquet(
            source=incoming_dir,
            target=target_dir,
            archive=archive_dir,
            issues=issues_dir,
            split=split,
        )

    except Exception as err:
        record_failure(
            name,
            "incoming-source compilation failed",
            err,
        )

        logger = getattr(
            processor,
            "logger",
            None,
        )

        if logger is not None:
            try:
                logger.exception(
                    "Incoming-source compilation failed for %s; "
                    "continuing with the next configured source",
                    name,
                )
            except Exception:
                pass

        continue

    print(
        f"✔ {name.upper()} completed.",
        flush=True,
    )


print(
    "\n" + "=" * 72,
    flush=True,
)

if processor_failures:
    print(
        "NRB processing completed with failures:",
        flush=True,
    )

    for processor_name, errors in processor_failures.items():
        for error in errors:
            print(
                f"  - {processor_name}: {error}",
                flush=True,
            )
else:
    print(
        "✔ All configured sources completed.",
        flush=True,
    )
# from pathlib import Path
# from processing.ae31 import AE31
# from processing.fidas import Fidas
# from processing.meteo import Meteo
# from processing.neph import Neph
# from processing.thermo import Thermo
# from processing.hmp110 import HMP110
# from processing.avo import AVO

# from toolbox.utils import load_config

# ##############################
# # Process NRB incoming files #
# ##############################
# # read configuration
# nrb = load_config(config_file="mch-nrb.yml")
# log_file = str(Path("logs") / "nrb.log")
# target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

# processors = {
#     "49i": Thermo(name="49i", log_file=log_file),
#     "fidas": Fidas(log_file=log_file),
#     "aurora3000": Neph(name="aurora3000", log_file=log_file),
#     "ae31": AE31(log_file=log_file),
#     "hmp110-ae31": HMP110(name="hmp110-ae31", log_file=log_file),
#     "hmp110-inlet": HMP110(name="hmp110-inlet", log_file=log_file),
#     "hmp110-lab": HMP110(name="hmp110-lab", log_file=log_file),
#     "avo-roof": AVO(name="avo-roof", log_file=log_file),    
#     "avo-garden": AVO(name="avo-garden", log_file=log_file),    
#     "avo-mogogosiek": AVO(name="avo-mogogosiek", log_file=log_file),    
#     "avo-huduma": AVO(name="avo-huduma", log_file=log_file),    
#     # "meteo": Meteo(name="vrxa00"),
# }

# for name, processor in processors.items():
#     print(f"▶ Processing {name.upper()} ...")
#     processor.compile_to_parquet(
#         source=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['incoming'] / name,
#         target=target / "level1" / "nrb",
#         archive=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['archive'] / name,
#         issues=Path(nrb['nrb-aq']['root']) / nrb['nrb-aq']['branches']['issues'] / name,
#         split=nrb['nrb-aq'][name]['split_parquet']
#     )    


▶ Processing 49I -> /product_data/data/pay/Kenya/git/gawkenyadata/level1/nrb
  Reading incoming source files: /product_data/data/pay/Kenya/NRB/incoming/49i
2026-09-07T06:42:50, INFO, 49i, Processing 49i-2026090705.zip
2026-09-07T06:42:50, INFO, 49i, 49i-2026090705.zip::49i-2026090705.csv: detected model=tei49i layout=modern header=True
2026-09-07T06:42:50, INFO, 49i, 49i-2026090705.zip::49i-2026090705.csv: extracted 58 row(s), 17 column(s)
2026-09-07T06:42:50, INFO, 49i, ✔ Written 58 new rows to /product_data/data/pay/Kenya/git/gawkenyadata/level1/nrb/2026/09/49i.parquet
2026-09-07T06:42:50, INFO, 49i, ✔ Parquet compilation completed. Total rows: 58
✔ 49I completed.

▶ Processing FIDAS -> /product_data/data/pay/Kenya/git/gawkenyadata/level1/nrb
  Reading incoming source files: /product_data/data/pay/Kenya/NRB/incoming/fidas
2026-09-07T06:42:50, INFO, fidas, Processing fidas-2026090705.zip
2026-09-07T06:42:50, INFO, fidas, ✔ Written 60 new rows to /product_data/data/pay/Kenya/git/gawke

In [3]:
from pathlib import Path
from processing.thermo import Thermo

from toolbox.utils import load_config

##############################
# Process BUC incoming files #
##############################
# read configuration
buc = load_config(config_file="mch-buc.yml")
target = Path("/product_data/data/pay/Kenya/git/gawkenyadata")

processors = {
    "49i": Thermo(name="49i"),
}

for name, processor in processors.items():
    print(f"▶ Processing {name.upper()} ...", flush=True)
    processor.compile_to_parquet(
        source=Path(buc['root']) / buc['branches']['incoming'] / name,
        target=target / "level1" / "buc",
        archive=Path(buc['root']) / buc['branches']['archive'] / name,
        issues=Path(buc['root']) / buc['branches']['issues'] / name,
        split=buc[name]['split_parquet']
    )

▶ Processing 49I ...
2026-09-07T06:42:57, WARNING, 49i, No valid data extracted.
